In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ---------- 2. 高效加载数据 ----------
# 使用适合的数据类型减少内存占用
dtypes = {
    'stock_id': 'int16',
    'date_id': 'int16',
    'seconds_in_bucket': 'int16',
    'imbalance_size': 'float32',
    'imbalance_buy_sell_flag': 'int8',
    'reference_price': 'float32',
    'matched_size': 'float32',
    'far_price': 'float32',
    'near_price': 'float32',
    'bid_price': 'float32',
    'bid_size': 'float32',
    'ask_price': 'float32',
    'ask_size': 'float32',
    'wap': 'float32',
    'target': 'float32'
}

train = pd.read_csv(
    '/kaggle/input/optiver-trading-at-the-close/train.csv',
    dtype=dtypes,
    parse_dates=False
)
print(f'完整数据加载完毕，形状: {train.shape}')

# ---------- 3. 按时间切分训练/验证 ----------
# 获取所有 date_id，用前 80% 训练，后 20% 验证
date_ids = sorted(train['date_id'].unique())
split_idx = int(len(date_ids) * 0.8)
train_dates = date_ids[:split_idx]
val_dates = date_ids[split_idx:]

print(f'训练日期数: {len(train_dates)} (从 {train_dates[0]} 到 {train_dates[-1]})')
print(f'验证日期数: {len(val_dates)} (从 {val_dates[0]} 到 {val_dates[-1]})')

train_mask = train['date_id'].isin(train_dates)
val_mask = train['date_id'].isin(val_dates)

X_train = train[train_mask].drop('target', axis=1).copy()
y_train = train.loc[train_mask, 'target'].copy()
X_val = train[val_mask].drop('target', axis=1).copy()
y_val = train.loc[val_mask, 'target'].copy()

del train  # 释放内存

print(f'训练集: {X_train.shape}, 验证集: {X_val.shape}')

# ---------- 4. 特征工程 ----------
def engineer_features(df):
    data = df.copy()
    
    # --- 价格特征 ---
    data['mid_price'] = (data['bid_price'] + data['ask_price']) / 2
    data['spread'] = data['ask_price'] - data['bid_price']
    data['wap_deviation'] = data['wap'] - data['mid_price']
    ref_price = data['reference_price'].fillna(data['mid_price'])
    data['price_deviation_from_ref'] = data['mid_price'] - ref_price
    
    # --- 订单簿不平衡 ---
    sum_vol = data['bid_size'] + data['ask_size']
    data['book_imbalance'] = np.where(
        sum_vol > 0,
        (data['bid_size'] - data['ask_size']) / sum_vol,
        0
    )
    data['log_bid_size'] = np.log1p(data['bid_size'])
    data['log_ask_size'] = np.log1p(data['ask_size'])
    
    # --- 拍卖簿特征 ---
    data['auction_pressure'] = data['far_price'] - data['near_price']
    data['auction_imbalance_signed'] = data['imbalance_size'] * data['imbalance_buy_sell_flag'].fillna(0)
    
    # --- 时序特征（分组计算）---
    data = data.sort_values(['stock_id', 'date_id', 'seconds_in_bucket'])
    
    def add_group_ts_features(group):
        group = group.sort_values('seconds_in_bucket')
        
        group['log_return'] = np.log(group['mid_price'] / group['mid_price'].shift(1))
        group['book_imb_rolling_mean_5'] = group['book_imbalance'].rolling(window=5, min_periods=1).mean()
        group['book_imb_rolling_std_5'] = group['book_imbalance'].rolling(window=5, min_periods=1).std()
        group['spread_rolling_mean_5'] = group['spread'].rolling(window=5, min_periods=1).mean()
        group['spread_rolling_std_5'] = group['spread'].rolling(window=5, min_periods=1).std()
        group['price_volatility_60'] = group['log_return'].rolling(window=60, min_periods=1).std()
        group['imbalance_rolling_mean_5'] = group['imbalance_size'].rolling(window=5, min_periods=1).mean()
        group['imbalance_change'] = group['imbalance_size'] - group['imbalance_size'].shift(1)
        
        # 滞后特征
        group['imbalance_size_lag1'] = group['imbalance_size'].shift(1)
        group['book_imbalance_lag1'] = group['book_imbalance'].shift(1)
        group['wap_lag1'] = group['wap'].shift(1)
        
        return group
    
    data = data.groupby('stock_id', group_keys=False).apply(add_group_ts_features)
    data = data.ffill().fillna(0)
    return data

print('开始特征工程（全量数据）...')
X_train_fe = engineer_features(X_train)
X_val_fe = engineer_features(X_val)
print('特征工程完成！')

# ---------- 5. 模型训练 ----------
feature_cols = [
    'seconds_in_bucket',
    'imbalance_size', 'imbalance_buy_sell_flag',
    'reference_price', 'matched_size', 'far_price', 'near_price',
    'bid_price', 'ask_price', 'bid_size', 'ask_size', 'wap',
    'mid_price', 'spread', 'wap_deviation', 'price_deviation_from_ref',
    'book_imbalance', 'log_bid_size', 'log_ask_size',
    'auction_pressure', 'auction_imbalance_signed',
    'log_return', 'book_imb_rolling_mean_5', 'book_imb_rolling_std_5',
    'spread_rolling_mean_5', 'spread_rolling_std_5',
    'price_volatility_60', 'imbalance_rolling_mean_5', 'imbalance_change',
    'imbalance_size_lag1', 'book_imbalance_lag1', 'wap_lag1'
]
feature_cols = [col for col in feature_cols if col in X_train_fe.columns]

X_train_model = X_train_fe[feature_cols]
X_val_model = X_val_fe[feature_cols]

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=50,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=100  # 每100轮打印一次进度
)

print('开始训练...')
model.fit(
    X_train_model, y_train,
    eval_set=[(X_val_model, y_val)],
    eval_metric='mae',
)

# ---------- 6. 评估 ----------
y_pred = model.predict(X_val_model)
mae = np.mean(np.abs(y_pred - y_val))
print(f'\n✅ 验证集 MAE: {mae:.6f}')

# 无脑基线
stock_mean = y_train.groupby(X_train['stock_id']).mean()
naive_pred = X_val['stock_id'].map(stock_mean).values
naive_mae = np.mean(np.abs(naive_pred - y_val))
print(f'📊 无脑基线 MAE: {naive_mae:.6f}')
print(f'🚀 提升: {(1 - mae/naive_mae)*100:.2f}%')

# 特征重要性
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print('\n📊 TOP 20 特征重要性：')
print(importance_df.head(20).to_string(index=False))

# 预测 vs 真实散点图
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_pred, alpha=0.1, s=1)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=1)
plt.xlabel('True Target')
plt.ylabel('Predicted Target')
plt.title('True vs Predicted (Full Data Baseline)')
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/optiver-trading-at-the-close/train.csv'